# Component 2 Walkthrough — Exceedance Probability Computation

C2 reads IFS ensemble forecasts from the IceChunk store (or any `xr.Dataset`),
computes rolling precipitation accumulations over multiple windows,
compares against GEV-fitted thresholds, and writes exceedance probabilities
to a Zarr store.

**Key modules**: `accumulations`, `exceedance`, `thresholds`, `writer`

In [ ]:
import numpy as np
import xarray as xr
import pandas as pd
from pathlib import Path
import tempfile
from datetime import date

WORK_DIR = Path(tempfile.mkdtemp(prefix='gik_c2_'))

NLAT, NLON, NMEMBERS, NSTEPS = 8, 8, 10, 16
LAT   = np.linspace(0.0, 4.0, NLAT, dtype=np.float32)
LON   = np.linspace(35.0, 39.0, NLON, dtype=np.float32)
STEPS = np.arange(0, NSTEPS * 6, 6, dtype=np.int32)
WINDOWS_H      = [24, 72, 168]
RETURN_PERIODS = [5, 20]
TEST_DATE      = date(2024, 10, 15)

rng = np.random.default_rng(42)
tp  = np.cumsum(
    rng.exponential(0.004, (NMEMBERS, NSTEPS, NLAT, NLON)).astype(np.float32),
    axis=1,
)

forecast_ds = xr.Dataset({'tp': xr.DataArray(
    tp,
    dims=['member', 'step', 'latitude', 'longitude'],
    coords={'member': np.arange(NMEMBERS), 'step': STEPS, 'latitude': LAT, 'longitude': LON},
    attrs={'units': 'm'},
)})
print(forecast_ds)

## 2.1  Rolling Accumulations

Converts cumulative TP (metres) into per-window accumulations for 24 h, 72 h, 168 h.

In [ ]:
from gik_icechain.exceedance.accumulations import compute_rolling_accumulations

acc = compute_rolling_accumulations(forecast_ds, windows_h=WINDOWS_H)
for w in WINDOWS_H:
    key  = f'tp_{w}h'
    vals = acc[key]
    print(f'{key}: shape={vals.shape}  min={float(vals.min()):.5f}  max={float(vals.max()):.4f} m')
    assert float(vals.min()) >= 0.0, f'{key} has negative values'

## 2.2  Adaptive GEV Thresholds

In [ ]:
from gik_icechain.exceedance.thresholds import AdaptiveGEVThresholds, ClimateMode, ENSOPhase, IODPhase, Season


def _make_thresholds(
    lat: np.ndarray,
    lon: np.ndarray,
    windows_h: list,
    return_periods: list,
) -> AdaptiveGEVThresholds:
    thr = AdaptiveGEVThresholds()
    template = xr.DataArray(
        np.full((len(lat), len(lon)), 0.001, dtype=np.float32),
        dims=['latitude', 'longitude'],
        coords={'latitude': lat, 'longitude': lon},
    )
    for season in Season:
        for enso in ENSOPhase:
            for iod in IODPhase:
                mode = ClimateMode(season, enso, iod)
                thr._thresholds[mode.key] = {
                    w: {rp: template * (rp / 5.0) for rp in return_periods}
                    for w in windows_h
                }
    return thr


thr_inst = _make_thresholds(LAT, LON, WINDOWS_H, RETURN_PERIODS)
print(f'Loaded {len(thr_inst._thresholds)} climate mode combinations.')

In [ ]:
zarr_path = str(WORK_DIR / 'thresholds.zarr')
thr_inst.save_zarr(zarr_path)

thr2 = AdaptiveGEVThresholds.load_zarr(zarr_path)
mode = ClimateMode(Season.OND, ENSOPhase.NEUTRAL, IODPhase.NEUTRAL)
np.testing.assert_allclose(
    thr_inst.get(24, 5, mode).values,
    thr2.get(24, 5, mode).values,
    rtol=1e-4,
)
print('Threshold Zarr round-trip — OK')

## 2.3  Exceedance Probabilities & Ensemble Confidence

In [ ]:
from gik_icechain.exceedance.exceedance import compute_exceedance_probabilities, compute_ensemble_confidence

day_results = {}
for w in WINDOWS_H:
    for rp in RETURN_PERIODS:
        thr = thr_inst.get(w, rp, mode)
        p   = compute_exceedance_probabilities(
            acc, xr.Dataset({f'rp_{rp}y': thr}), w, rp, member_dim='member'
        )
        day_results[(w, rp)] = p
        assert float(p.min()) >= 0.0 and float(p.max()) <= 1.0
        print(f'  ({w}h, {rp}yr): min={float(p.min()):.3f}  max={float(p.max()):.3f}')

conf = compute_ensemble_confidence(acc, window_h=24, member_dim='member')
unique_conf = set(int(v) for v in np.unique(conf.values))
assert unique_conf.issubset({0, 1, 2})
print(f'Ensemble confidence states: {sorted(unique_conf)} — OK')

## 2.4  Writing to Zarr

In [ ]:
from gik_icechain.exceedance.writer import build_exceedance_dataset, write_exceedance_store

exc_da  = build_exceedance_dataset(day_results, TEST_DATE)
conf_da = conf.assign_coords(date=pd.Timestamp(TEST_DATE)).expand_dims('date')

output_uri = str(WORK_DIR / 'exceedance.zarr')
write_exceedance_store(
    {TEST_DATE: exc_da},
    output_uri,
    confidence_dict={TEST_DATE: conf_da},
    endpoint_url=None,
)

ds = xr.open_zarr(output_uri, consolidated=False)
print(ds)
assert 'exceedance_prob' in ds
assert 'ensemble_confidence' in ds
p_vals   = ds['exceedance_prob'].values
p_finite = p_vals[np.isfinite(p_vals)]
assert p_finite.min() >= 0.0 and p_finite.max() <= 1.0
print('Exceedance Zarr written and validated — OK')